# Music Genre Classification: Project Walkthrough

Explore spectrogram preprocessing, the CNN architecture, training, and evaluation. Install the package from the repository root with `python -m pip install -e .` before opening this notebook. Download the dataset separately and train a checkpoint to run predictions.

## 1. What does the network see?

A spectrogram maps time horizontally and frequency vertically. Each source image produces 32 resized RGB slices. The model predicts one of ten genres. These slices are correlated: split **source tracks first**, then create slices within each partition.

In [ ]:
from music_genre.data import GENRES
from music_genre.model import GenreCNN
import torch
print(GENRES)
model = GenreCNN()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
model.eval()
with torch.inference_mode():
    print("Output shape:", tuple(model(torch.zeros(2, 3, 64, 64)).shape))

## 2. Reproducible experiment

From the repository root, run `music-genre prepare`, `music-genre train`, and `music-genre evaluate` using the commands in the README. The manifest freezes source-track partitions and hashes. Training selects the best validation track macro-F1; held-out test metrics are computed separately.

## 3. Inspect a completed run

After training and evaluating, point `run` below at your output folder to inspect slice and track metrics.

In [ ]:
from pathlib import Path
import json
run = Path("../runs/cnn")  # adjust if your notebook working directory differs
if (run / "test.json").exists():
    result = json.loads((run / "test.json").read_text())
    print("Slice metrics:", result["slice"]["accuracy"], result["slice"]["macro_f1"])
    print("Track metrics:", result["track"]["accuracy"], result["track"]["macro_f1"])
else:
    print("No completed real-data run is present. Follow README.md to create one.")

## 4. Interpret the reported results

The project records **69.4% GTZAN test accuracy** and **65.4% external accuracy**, rounded to one decimal place. The GTZAN score uses a slice-level split that can share tracks between partitions, with dropout potentially active during scoring. The Random Forest uses a different split. The package evaluates by source track; no GTZAN score has yet been recorded for that protocol. External assets are not bundled. See `../docs/evaluation.md` for metric definitions and interpretation.

## 5. Next experiment

Fix rendering and track partitions, train both models, compare track macro-F1 and per-class errors, then repeat across seeds. Check duplicate recordings and artist overlap before drawing stronger generalization conclusions.